# **<font color="red">Part1: Introduction to ADK Gemini Live API Toolkit</font>**
-  ADK provides a production-ready framework for building Bidi-Streaming applications with Gemini models.
- Bidi-Streaming, the underlying Live API technology(**Gemini Live API** and **Vertex AI Live API**) `LiveRequestQueue`, `Runner`, `Agent` and Complete **FastAPI** implementation.
- **Features:**
  - ***WebSocket Communication:*** Real-time bidirectional streaming with concurrent upstream/downstream tasks.
  - ***Multimodel Requests:*** Text, audio, and image/video input with automatic transcription.
  - ***Flexible Responses:*** Text or audio output based on model capabilities.
  - ***Interactive UI:*** Web interface with event console for monitoring Live API events
  - ***Google Search Integration:*** Agent equipped with tool calling capabilities
- **Bidi-Streaming:**
  - Real-time, two-way communication. Where both human and AI can speak, listen, and response simultaneously.
  - **Key Characteristics:**
    - Two-way Communicaiton
    - Responsive Interruption
    - Best for Multimodal
- **Streaming Types:**
  - **Server-Side Streaming:** One-way data flow from serer to client. Like watching a live video stream-you receive continuous data but can not interact with it in real-time.
  - **Token-Level Streaming:** Sequential text token delivery without interruption. The AI generates response word-by-word, but you must wait for completion before sending new input.
  - **Bidi-Streaming:** Full two-way communication with interruption support. True conversational AI where both parties can speak, listen, and response simultaneously.


## **<font color="blue">Gemini Live API and Vertex AI Live API</font>**
ADK Gemini Live API Toolkit capabilities are powered by Live API technology, available through two platforms: **Gemini Live API**(via Google AI Studio) and **Vertex AI Live API**(via Google Cloud). Both provide real-time, low-latency streaming conversations with Gemini models, but serve different development and deployment needs.

### **What is the Live API?**
- Live API is Google's real-time conversational AI technology that enables low-latency Bidi-Streaming with Gemini models
- **Core Capabilities:**
  - **Multimodel streaming**
  - **Voice Activity Detection(VAD)**
  - **Immediate Responses**
  - **Intelligent Interruption**
  - **Audio Transcription**
  - **Session Management**
  - **Tool Integration**
- **Native Audio Model Features:**
  - **Proactive Audio**
  - **Affective Dialog**
- **Technical Specifications:**
  - **Audio Input :** 16-bit PCM at 16kHz (mono)
  - **Audio Output:** 16-bit PCM at 24kHz (native audio models)
  - **Video Input:** 1 frame per second, recommended 768x768 resolution
  - **Context Windows:** Varies by model (typically 32k-128k tokens for Live API models)
  - **Languages:** 24+ languages supported with automatic detection


## **<font color="blue">ADK Gemini Live API Toolkit Application Lifecycle</font>**
- ADK Gemini Live API Toolkit integrates Live API session into the ADK framework's application lifecycle. This integration creates a four-phase lifecycle that combines ADK's agent management with Live API's real-time streaming capabilities:
- #### **Phase 1: Application Initialization** (Once at Startup)
  - ADK Application Initialization
    - Create an `Agent`
    - Create a `SessionService`
    - Create a `Runner`
- #### **Phase 2: Session Initialization** (Once per User Session)
  - ADK `Session` initialization
  - ADK Gemini Live API ToolKit Initialization:
    - Create a `RunConfig` for configuring ADK Gemini Live API Toolkit
      ```python
        from google.genai import types
        from google.adk.agents.run_config import RunConfig, StreamingMode
        
        # Configuration types are accessed via types module
        run_config = RunConfig(
            session_resumption=types.SessionResumptionConfig(),
            context_window_compression=types.ContextWindowCompressionConfig(...),
            speech_config=types.SpeechConfig(...),
            # etc.
        )
    - Start a `run_live()` event loop
- #### **Phase 3: Bidi-Streaming with `run_live()` event loop** (One or More Times per User Session)
  - Upstream: User sends message to the agent with `LiveRequestQueue`
  - Downstream: Agent responds to the user with `Event`
- #### **Phase 4: Terminate Live API session (One or More Times per User Session)
  - `LiveRequestQueue.close()`

- #### **Lifecycle Flow Overview:**
```
                        Phase 1: Application init
                             Once at Startup
                                    |
                                    |
                                   \|/
                        Phase 2: Session init
                        Per User Connection
                                    |
                                    |
                                    ^
                                   / \
                   _______________/   \_______________
                  |                                   |
            Phase 3: Bidi-Streaming              New Connection
              Active Communication                    |
                  |                                   |
                  |                                   |
                  |                                   |
                  |_______ Phase 4: Terminate  _______|  
                              Close Session
```


In [1]:
import os
import asyncio
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
import google.genai.types as types
from config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "live_chatbot"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = config.MODEL

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Create LlmAgent (chatbot)
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="ChatAgent",
    instruction="""
    You are a helpful assistant. Respond politely and concisely to user questions.
    """
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# Create a new session
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# -----------------------------
# Setup Runner
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service
)

# -----------------------------
# Chat function
# -----------------------------
async def chat(user_input):
    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )
    
    events = runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
    
    async for event in events:
        # Print final response only
        if event.is_final_response() and event.content and event.content.parts:
            text = event.content.parts[0].text
            print("ChatAgent:", text)

# -----------------------------
# Chat
# -----------------------------
await chat("Hello! Can you suggest a fun weekend activity?")
await chat("Give me a short summary of Artificial Intelligence.")


ChatAgent: Hello! How about a picnic in a local park?
ChatAgent: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines programmed to think like humans and mimic their actions. It encompasses machine learning, natural language processing, and problem-solving.


In [5]:
import os
import asyncio

from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent, LiveRequestQueue
from google.adk.agents.run_config import RunConfig, StreamingMode

import google.genai.types as types
from config import config


# -----------------------------
# CONFIGURATION
# -----------------------------
APP_NAME = "live_chatbot"
USER_ID = "user_001"
SESSION_ID = "session_001"
LIVE_MODEL = config.LIVE_MODEL

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

print("Using model:", LIVE_MODEL)


# -----------------------------
# CREATE AGENT
# -----------------------------
root_agent = LlmAgent(
    model=LIVE_MODEL,
    name="ChatAgent",
    instruction="You are a helpful assistant. Respond politely and concisely."
)


# -----------------------------
# SERVICES
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()


# -----------------------------
# INITIALIZE SESSION
# -----------------------------
async def init_session():
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )


# -----------------------------
# RUNNER
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service
)


# -----------------------------
# LIVE CHAT FUNCTION
# -----------------------------
async def live_chat():
    live_queue = LiveRequestQueue()

    # RunConfig
    run_config = RunConfig(
        streaming_mode=StreamingMode.BIDI,
        response_modalities=[types.Modality.TEXT],
        session_resumption=types.SessionResumptionConfig()
    )

    # -----------------------------
    # RECEIVE STREAM
    # -----------------------------
    async def receive():
        async for event in runner.run_live(
            user_id=USER_ID,
            session_id=SESSION_ID,
            live_request_queue=live_queue,
            run_config=run_config
        ):
            try:
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if hasattr(part, "text") and part.text:
                            print("ChatAgent:", part.text)
            except Exception as e:
                print("Error processing event:", e)

    receive_task = asyncio.create_task(receive())

    # -----------------------------
    # SEND LOOP
    # -----------------------------
    try:
        while True:
            user_input = input("You: ")

            if user_input.lower() in ["exit", "quit"]:
                break
            
            await live_queue.send(
                types.Content(
                    role="user",
                    parts=[types.Part(text=user_input)]
                )
            )

    except KeyboardInterrupt:
        print("\nExiting...")

    # -----------------------------
    # CLEANUP
    # -----------------------------
    await live_queue.close()
    await receive_task


# -----------------------------
# MAIN
# -----------------------------
async def main():
    await init_session()
    await live_chat()


# -----------------------------
# ENTRY POINT
# -----------------------------
if __name__ == "__main__":
    try:
        # Normal Python
        asyncio.run(main())
    except RuntimeError:
        # Jupyter fallback
        await main()

Using model: gemini-2.0-flash-live-001


You:  Hello


TypeError: object NoneType can't be used in 'await' expression

An unexpected error occurred in live flow: 1008 None. models/gemini-2.0-flash-live-001 is not found for API version v1alpha, or is not supported for bidiGenerateContent. Call Li
Traceback (most recent call last):
  File "D:\Agent-Development-Kit\venv\Lib\site-packages\google\genai\live.py", line 1105, in connect
    raw_response = await ws.recv(decode=False)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Agent-Development-Kit\venv\Lib\site-packages\websockets\asyncio\connection.py", line 322, in recv
    raise self.protocol.close_exc from self.recv_exc
websockets.exceptions.ConnectionClosedError: received 1008 (policy violation) models/gemini-2.0-flash-live-001 is not found for API version v1alpha, or is not supported for bidiGenerateContent. Call Li; then sent 1008 (policy violation) models/gemini-2.0-flash-live-001 is not found for API version v1alpha, or is not supported for bidiGenerateContent. Call Li

During handling of the above exception, another exception occurred:
